# GVHMR 추론 — 원클릭 자동화 노트북

`motion-retarget-poc`의 핵심 기술 1(영상 → 3D 모션 추정) 단계. 스파이크에서 실제로 검증된 셀들을 그대로 옮겨왔다.

## 이 노트북에서 사람이 직접 해야 하는 부분은 딱 2곳뿐이다

1. **GPU 런타임 선택** (아래 셀 1을 돌리기 전에) — 상단 메뉴 `런타임 > 런타임 유형 변경` → GPU 선택. API로 대신 할 수 없다.
2. **SMPL/SMPLX 체크포인트 업로드** (아래 "사람이 할 차례" 셀 부근) — smpl.is.tue.mpg.de / smpl-x.is.tue.mpg.de에서 개인 계정으로 회원가입 후 받은 파일 2개를 업로드. 라이선스 동의가 이메일 인증을 요구하는 구조라 자동화가 불가능하다. **최초 1회만 하면 되고, 받은 파일은 본인 Google Drive 등에 보관해두면 다음 실행부터는 업로드만 하면 된다.**

이 2곳을 빼면 아래 셀을 위에서 아래로 그냥 순서대로 실행하면 된다 (`런타임 > 모두 실행`).

## 0. GPU 확인

아래 셀에서 GPU가 안 잡히면(`nvidia-smi` 에러) 상단 `런타임 > 런타임 유형 변경`에서 GPU를 고르고 다시 실행.

In [ ]:
!nvidia-smi

## 1. Python 3.10 환경 구성 (자동)

GVHMR의 `pytorch3d` 의존성이 Python 3.10 전용 wheel로 고정돼 있어서, Colab 기본 Python(3.13)이 아닌 conda 3.10 환경을 만든다.

> `condacolab.install()` 실행 시 커널이 1회 자동 재시작된다 — 정상 동작이니 놀라지 말 것. 재시작 후 바로 다음 셀부터 이어서 실행하면 된다.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()
!conda create -n gvhmr python=3.10 -y -q

## 2. GVHMR 클론 + 의존성 설치 (자동)

`chumpy`는 기본 pip build-isolation에서 실패하는 게 확인된 패키지라, numpy를 먼저 깔고 `--no-build-isolation`으로 따로 설치한다 (스파이크에서 확인한 우회법).

In [ ]:
%cd /content
!git clone -q https://github.com/zju3dv/GVHMR.git
%cd GVHMR
PY = "/usr/local/envs/gvhmr/bin/python"
PIP = "/usr/local/envs/gvhmr/bin/pip"

In [ ]:
!/usr/local/envs/gvhmr/bin/pip install -q numpy==1.23.5 setuptools==68.0.0 wheel
!/usr/local/envs/gvhmr/bin/pip install -q --no-build-isolation chumpy
!/usr/local/envs/gvhmr/bin/pip install -q -r requirements.txt
!/usr/local/envs/gvhmr/bin/pip install -q -e .

In [ ]:
!/usr/local/envs/gvhmr/bin/python -c "import torch, hmr4d; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available()); print('hmr4d OK')"

## 3. 체크포인트 다운로드 (자동)

> 공식 문서가 안내하는 Google Drive 링크는 **공유 쿼터 초과로 막혀 있는 경우가 많다** (스파이크에서 직접 재현 확인). 대신 커뮤니티가 올려둔 HuggingFace 미러(`camenduru/GVHMR`)를 쓴다 — 더 빠르고 안정적이다.

In [ ]:
!/usr/local/envs/gvhmr/bin/pip install -q huggingface_hub
!/usr/local/envs/gvhmr/bin/hf download camenduru/GVHMR \
  "gvhmr/gvhmr_siga24_release.ckpt" \
  "hmr2/epoch=10-step=25000.ckpt" \
  "vitpose/vitpose-h-multi-coco.pth" \
  "dpvo/dpvo.pth" \
  --local-dir /content/GVHMR/inputs/checkpoints

In [ ]:
# YOLO는 GVHMR의 Drive 사본 대신 Ultralytics 공식 배포에서 직접 받는다 (쿼터 문제 없음)
!mkdir -p /content/GVHMR/inputs/checkpoints/yolo
!/usr/local/envs/gvhmr/bin/python -c "from ultralytics import YOLO; YOLO('yolov8x.pt')"
!mv /content/GVHMR/yolov8x.pt /content/GVHMR/inputs/checkpoints/yolo/yolov8x.pt

## 4. 사람이 할 차례 — SMPL / SMPL-X 바디 모델

아래 두 파일이 **자동으로 준비되지 않는 유일한 것**이다. 각자 계정으로 받아서 이 Colab 세션에 업로드해야 한다.

> ⚠️ **SMPL과 SMPL-X는 완전히 다른 두 개의 사이트/모델이다.** 이름이 비슷해서 헷갈리기 쉬운데, `smpl-x.is.tue.mpg.de`(X 있음)에서는 SMPL-X 파일만 받을 수 있고 `smpl.is.tue.mpg.de`(X 없음)에서는 SMPL 파일만 받을 수 있다. **두 사이트에 각각 따로 회원가입해야 한다** (실제로 이 둘을 혼동해서 SMPL-X 파일을 `smpl/` 폴더에 잘못 넣는 실수가 스파이크 중 실제로 발생했다).

1. [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) (X **없는** 사이트) 로그인(최초 1회 회원가입) → Download → **"SMPL for Python"** 계열 zip → 압축 풀면 나오는 neutral 모델 파일(`basicmodel_neutral_lbs_...pkl` 등)을 **`SMPL_NEUTRAL.pkl`로 이름 바꿔서** 아래 경로에 업로드.
2. [smpl-x.is.tue.mpg.de](https://smpl-x.is.tue.mpg.de/) (X **있는** 사이트) 로그인(최초 1회 회원가입) → Download → **"SMPL-X v1.1 (NPZ+PKL) — SMPL-X 파이썬 코드베이스에 사용하세요"** zip → 압축 풀면 나오는 **`SMPLX_NEUTRAL.npz`**를 그대로 아래 경로에 업로드.

(UV맵, VPoser, Homogenus, Blender 애드온, Unity 패키지, 줄리아 모델스, SMPL-X 2020 등 나머지 다운로드 항목은 전부 불필요.)

업로드 경로 (왼쪽 파일 탐색기에서 폴더 만들고 드래그 앤 드롭):
- `/content/GVHMR/inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl`
- `/content/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz`

업로드가 끝나면 아래 확인 셀을 돌려서 파일이 있는지, **그리고 실제로 로드되는지**까지 체크한다 (파일명만 맞고 내용물이 잘못됐거나 업로드 중 깨진 경우를 미리 걸러낸다).

In [ ]:
import os
smpl_path = "/content/GVHMR/inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl"
smplx_path = "/content/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz"
for p in [smpl_path, smplx_path]:
    print("OK " if os.path.exists(p) else "누락 ", p)
assert os.path.exists(smpl_path) and os.path.exists(smplx_path), "위 두 파일을 먼저 업로드하세요"

In [ ]:
# 파일이 진짜 로드되는지까지 확인 (SMPL-X 파일을 잘못 넣었거나 업로드 중 손상된 경우 여기서 바로 걸러진다)
import sys
sys.path.insert(0, "/content/GVHMR")
from hmr4d.utils.body_model.smpl_lite import SmplLite
from hmr4d.utils.body_model.smplx_lite import SmplxLite
SmplLite(model_path="/content/GVHMR/inputs/checkpoints/body_models/smpl", gender="neutral")
SmplxLite(model_path="/content/GVHMR/inputs/checkpoints/body_models/smplx", gender="neutral")
print("SMPL / SMPL-X 둘 다 정상 로드됨")

## 5. 추론 실행 (자동)

GVHMR에 내장된 샘플 영상(`inputs/demo/dance_3.mp4`)으로 먼저 전체 파이프라인이 도는지 확인한다. 우리 영상을 쓰려면 `--video` 경로만 바꾸면 된다.

In [ ]:
%cd /content/GVHMR
!/usr/local/envs/gvhmr/bin/python tools/demo/demo.py --video inputs/demo/dance_3.mp4 -s

결과는 `outputs/demo/dance_3/` 아래에 생긴다. 여기서 나온 3D 모션 결과(`hmr4d_results.pt`)를 다음 단계인 Blender 리타겟팅(`blender/retarget_render.py`)에 넘긴다.